# DiariZen-Large — standalone runner

DiariZen cannot share a session with the main notebook, so it runs here and hands
over plain RTTM files that `diarization.import_external_rttm()` adopts. One
scoring implementation, one set of metrics, one explorer.

**Why a separate session, precisely.** `requirements.txt` pins `numpy==1.26.4`
while `pyannote.audio` 4.x pulls numpy 2.x. That is the whole conflict. Two
things that are *not* problems, despite what the README implies:

* **Python.** `pyproject.toml` says `requires-python = ">=3.10"`, so Kaggle's
  3.11 is fine. The README's "Python 3.10" is what the authors used.
* **pyannote-audio.** It is a plain vendored directory in the repo, not a git
  submodule, so an ordinary `git clone` already has it. Only `dscore` is a
  submodule and we do not need it — we score with our own harness.

**The trap.** `pyproject.toml` declares *no* dependencies at all, so
`pip install -e .` installs the package and nothing else. The real dependency
list is `requirements.txt`, and it has to be installed explicitly.

**The other trap.** The README installs the vendored pyannote with
`-c ../constraints.txt`, which pins `torch==2.1.1+cu121`. On Kaggle that
replaces the preinstalled torch 2.10+cu128 — a ~2.5 GB download and a real risk
of a CUDA mismatch on the T4. We install it unconstrained and keep the working
torch. If DiariZen turns out to need the old torch, the smoke gate below catches
it in ten minutes rather than three hours.

## Before running

| Setting | Value |
|---|---|
| **Internet** | **ON** — the clone and pip both need it |
| **Accelerator** | **GPU T4** |
| **Input** | the same audio dataset the main notebook uses |

Then **Save Version** when the sweep finishes, or `/kaggle/working` is wiped and
all of it re-runs.


In [ ]:
# --- install ---------------------------------------------------------------
# Order matters. requirements.txt carries the real dependency list (pyproject
# declares none), and it is what pins numpy to 1.26.4. The vendored
# pyannote-audio goes in WITHOUT -c constraints.txt so the session keeps its
# existing torch -- see the header for why.
import subprocess, sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
SRC = Path("/kaggle/working/DiariZen") if IN_KAGGLE else Path("./DiariZen")

def sh(*a):
    r = subprocess.run(a, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise SystemExit(f"failed: {' '.join(a[:6])}...")
    return r

if not SRC.exists():
    sh("git", "clone", "-q", "--depth", "1",
       "https://github.com/BUTSpeechFIT/DiariZen.git", str(SRC))
    print("cloned", SRC)

pip = [sys.executable, "-m", "pip", "install", "-q"]
sh(*pip, "-r", str(SRC / "requirements.txt"))   # numpy==1.26.4, accelerate==1.6.0, ...
sh(*pip, "-e", str(SRC))                        # the package itself
sh(*pip, "-e", str(SRC / "pyannote-audio"))     # vendored 3.1, unconstrained
print("installed")

# numpy was almost certainly downgraded under a running interpreter, which
# leaves this kernel holding the old module object. Same check, and same
# reason, as cell 1.0 of the main notebook.
_loaded = __import__("numpy").__version__
_disk = subprocess.run([sys.executable, "-c", "import numpy;print(numpy.__version__)"],
                       capture_output=True, text=True).stdout.strip()
if _disk and _disk != _loaded:
    print("=" * 70)
    print("RESTART THE KERNEL, then run this cell again (it will be quick).")
    print(f"  numpy: loaded {_loaded}, on disk {_disk}")
    print("  Run > Restart & clear cell outputs")
    print("=" * 70)
    raise SystemExit("restart required")
print("numpy", _loaded, "-- ready")


## Paths

`AUDIO_DIR` is the same audio the main notebook uses. `OUT_DIR` receives one
`<clip_id>.rttm` per clip — that folder is the entire handoff.

`PREV_RTTM` is the previous session's output attached as an Input. Kaggle wipes
`/kaggle/working`, so without it a second sitting starts from zero.


In [ ]:
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()

# Same audio the main notebook uses -- check the Input panel for the slug.
AUDIO_DIR = Path("/kaggle/input/dataset/audio_16k") if IN_KAGGLE else Path("local_out/audio_16k")
OUT_DIR   = Path("/kaggle/working/diarizen_rttm") if IN_KAGGLE else Path("./diarizen_rttm")
# A previous session's RTTMs, attached as an Input after Save Version. The
# sweep is resumable, but only if it can SEE what the last session finished.
PREV_RTTM = Path("/kaggle/input/diarizen-rttm") if IN_KAGGLE else None

OUT_DIR.mkdir(parents=True, exist_ok=True)
if PREV_RTTM and PREV_RTTM.exists():
    import shutil
    n = 0
    for f in PREV_RTTM.rglob("*.rttm"):
        dst = OUT_DIR / f.name
        if not dst.exists():
            shutil.copyfile(f, dst); n += 1
    print(f"resumed {n} RTTMs from {PREV_RTTM}")

wavs = sorted(AUDIO_DIR.glob("*.wav"))
assert wavs, f"no wavs under {AUDIO_DIR} -- attach the audio dataset and fix AUDIO_DIR"
todo = [w for w in wavs if not (OUT_DIR / f"{w.stem}.rttm").exists()]
print(f"{len(wavs)} clips, {len(wavs)-len(todo)} already done, {len(todo)} to run  ->  {OUT_DIR}")


In [ ]:
# --- load, then SMOKE GATE before committing to a multi-hour sweep ----------
import time, torch, soundfile as sf
from diarizen.pipelines.inference import DiariZenPipeline

MODEL = "BUT-FIT/diarizen-wavlm-large-s80-md-v2"
pipe = DiariZenPipeline.from_pretrained(MODEL)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    pipe.to(DEV)
    print("GPU:", torch.cuda.get_device_name(0))
print("loaded", MODEL, "on", DEV, "| torch", torch.__version__)


def to_rttm(clip_id, ann):
    """The RTTM shape the main pipeline writes: start + DURATION, and no spaces
    in the speaker label, since RTTM is whitespace-delimited."""
    out = []
    for seg, _, label in ann.itertracks(yield_label=True):
        if seg.end <= seg.start:
            continue
        spk = "_".join(str(label).split())
        out.append(f"SPEAKER {clip_id} 1 {seg.start:.3f} {seg.end - seg.start:.3f} "
                   f"<NA> <NA> {spk} <NA> <NA>")
    return "\n".join(out) + ("\n" if out else "")


# Three clips spanning the duration range, so RTF is measured rather than
# assumed and fixed per-call overhead is visible against streaming cost.
smoke = sorted(wavs, key=lambda w: sf.info(str(w)).duration)
smoke = [smoke[0], smoke[len(smoke)//2], smoke[-1]]
rows = []
for w in smoke:
    dur = sf.info(str(w)).duration
    t = time.time(); ann = pipe(str(w)); el = time.time() - t
    turns = [(s.start, s.end, l) for s, _, l in ann.itertracks(yield_label=True)]
    spk = len({l for _, _, l in turns})
    rows.append((w.stem, dur, el, el/dur, len(turns), spk,
                 max((e for _, e, _ in turns), default=0.0)))
    (OUT_DIR / f"{w.stem}.rttm").write_text(to_rttm(w.stem, ann), encoding="utf-8")

print(f"\n{'clip':<26}{'dur':>8}{'elapsed':>9}{'rtf':>7}{'turns':>7}{'spk':>5}{'max_end':>9}")
for r in rows:
    print(f"{r[0]:<26}{r[1]:8.1f}{r[2]:9.1f}{r[3]:7.3f}{r[4]:7d}{r[5]:5d}{r[6]:9.1f}")

# Abort criteria, fixed in advance so the decision is not made under sunk cost.
problems = []
if all(r[5] <= 1 for r in rows):
    problems.append("every clip returned <=1 speaker -- the pipeline is not diarizing")
if any(r[4] == 0 for r in rows):
    problems.append("a clip returned no turns at all")
for r in rows:
    if r[6] > r[1] + 0.5:
        problems.append(f"{r[0]}: turns run past the audio ({r[6]:.1f}s > {r[1]:.1f}s)")
if problems:
    print("\nSMOKE GATE FAILED:")
    for p in problems:
        print("  -", p)
    raise SystemExit("do not start the sweep")

mean_rtf = sum(r[3] for r in rows) / len(rows)
total = sum(sf.info(str(w)).duration for w in wavs)
todo_sec = sum(sf.info(str(w)).duration for w in todo)
print(f"\nmean RTF {mean_rtf:.3f} (spread {min(r[3] for r in rows):.3f}-{max(r[3] for r in rows):.3f})")
print(f"projected: {mean_rtf*total/3600:.1f} h for all {len(wavs)} clips, "
      f"{mean_rtf*todo_sec/3600:.1f} h for the {len(todo)} remaining")
print("Kaggle GPU sessions cap at 12 h. If the projection is close to that, run"
      " it in two sittings -- the next cell resumes from whatever is on disk.")


In [ ]:
# --- full sweep ------------------------------------------------------------
# Resumable: an existing RTTM is a completed clip. Written via a .part file and
# os.replace so a session killed mid-write cannot leave a truncated RTTM that
# the resume logic would then treat as done.
import os, time

todo = [w for w in wavs if not (OUT_DIR / f"{w.stem}.rttm").exists()]
done = failed = 0
t0 = time.time()
for i, wav in enumerate(todo, 1):
    dst = OUT_DIR / f"{wav.stem}.rttm"
    try:
        t = time.time()
        ann = pipe(str(wav))
        tmp = dst.with_suffix(".rttm.part")
        tmp.write_text(to_rttm(wav.stem, ann), encoding="utf-8")
        os.replace(tmp, dst)
        done += 1
        el = time.time() - t
        print(f"[{i}/{len(todo)}] {wav.stem}  {el:.1f}s  "
              f"{len({l for _,_,l in ann.itertracks(yield_label=True)})} spk  "
              f"| elapsed {(time.time()-t0)/60:.0f}m")
    except Exception as exc:
        failed += 1
        print(f"[{i}/{len(todo)}] {wav.stem}  FAILED {type(exc).__name__}: {exc}")

have = len(list(OUT_DIR.glob("*.rttm")))
print(f"\n{done} written, {failed} failed, {have}/{len(wavs)} clips now have an RTTM "
      f"in {(time.time()-t0)/60:.1f} min")
if have < len(wavs):
    print("INCOMPLETE -- Save Version, attach this output as PREV_RTTM, and re-run.")


## Hand off to the main pipeline

**Save Version first.** Then attach this notebook's output to `main_kaggle.ipynb`
as an Input and, after cell 2.2:

```python
from sarvam_diar import diarization
diarization.import_external_rttm(
    cfg,
    "/kaggle/input/<this-notebook-output>",       # or a local path
    model="diarizen-large",
    clip_durations={c.clip_id: c.duration for c in inputs.values()},
)
```

`import_external_rttm` re-writes each RTTM through our own writer and drops the
same sidecar a native run would, so `is_done()`, scoring, the rankings and the
error explorer treat `diarizen-large` exactly like a model we ran in-process.
Re-run the scoring and export cells and it appears as a fourth column.

Nothing about the ground truth reaches this notebook — it reads audio and writes
turns, which is the same contract every other model in the roster has.


In [ ]:
import shutil, os
zip_path = shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR)
print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB, "
      f"{len(list(OUT_DIR.glob('*.rttm')))} rttm files)")
print("\nDownload from the Output tab, or Save Version and attach it to the")
print("main notebook as an Input.")